# N9.1 — Deep Agents: contexto, ferramentas e delegação

* Professor: Julio Cesar dos Reis <a href="mailto:dosreis@unicamp.br">(dosreis@unicamp.br)</a>
* Código base: Lucas
* Monitor: Alejandro Núñez Arroyo <a href="mailto:r299215@dac.unicamp.br">(r299215@dac.unicamp.br)</a>

Um agente de tool calling dá conta de tarefas de poucos passos: o modelo pede uma ferramenta, lê o
resultado e responde. Em tarefas longas ele esbarra na janela de contexto. Vinte chamadas com resultados
grandes enchem a janela antes de a tarefa acabar, e o começo da conversa sai fora junto.

Deep agents são esse mesmo loop com quatro peças em volta: um plano explícito, um sistema de arquivos,
subagentes e instruções carregadas sob demanda. Todas servem para tirar da janela o que não precisa estar
nela agora.

O notebook roda nove casos sobre a mesma base de dados (Chinook, uma loja de música). Cada caso liga uma
peça e mostra o que ela produz:

| # | Caso | Peça do harness | Seção |
|---|---|---|---|
| 1 | Analista SQL | ferramentas próprias | §3 |
| 2 | Duas conversas | threads e checkpointer | §4 |
| 3 | Conversa que estoura a janela | resumo e offloading | §5 |
| 4 | Consultas encadeadas | interpretador e PTC | §6 |
| 5 | Arquivo de referência protegido | backends e permissões | §7 |
| 6 | Preferência que sobrevive à thread | memória de longo prazo | §8 |
| 7 | Qualificar um lead | skills | §9 |
| 8 | Cadastrar um cliente | human-in-the-loop | §10 |
| 9 | Orçamento revisado | subagentes | §11 |

## 0.1 Pré-requisitos

- agente com tool calling e o loop de raciocínio;
- LangChain e LangGraph: `invoke`, mensagens, `thread_id`;
- Python: decoradores, `pathlib`, funções assíncronas;
- SQL básico (`SELECT`, `JOIN`, `GROUP BY`).

Os casos vêm dos notebooks [Deep Agents](https://academy.langchain.com/courses/foundation-introduction-to-deepagents) da LangChain
Academy.

## 0.2 Objetivos da aula

Ao final deste notebook, você deverá saber:

1. Descrever o que `create_deep_agent` acrescenta a um agente de tool calling comum.
2. Separar as três camadas do prompt: instruções da SDK, instruções suas e memória carregada.
3. Escrever uma ferramenta que devolve o erro como dado em vez de levantar exceção.
4. Explicar o que o `checkpointer` guarda e o que o `thread_id` separa.
5. Identificar quem escreve na janela de contexto e como resumo e offloading a esvaziam.
6. Decidir entre encadear consultas no modelo ou dentro de um `eval`.
7. Escolher um backend de arquivos e negar uma escrita por código, não por prompt.
8. Distinguir memória de thread de memória de longo prazo pelo escopo.
9. Empacotar um procedimento como skill e explicar por que ele não ocupa contexto até ser usado.
10. Colocar uma aprovação humana antes de uma escrita e tratar as quatro decisões.
11. Justificar uma delegação por isolamento de contexto e por isolamento de permissões.

## 0.3 Mapa do notebook

1. O que o harness acrescenta ao loop.
2. O prompt do sistema em camadas.
3. Caso 1 - ferramentas próprias.
4. Caso 2 - turnos, threads e checkpointer.
5. Caso 3 - a janela de contexto.
6. Caso 4 - interpretador e chamadas programáticas.
7. Caso 5 - sistema de arquivos e permissões.
8. Caso 6 - memória entre threads.
9. Caso 7 - skills.
10. Caso 8 - human-in-the-loop.
11. Caso 9 - delegação a subagentes.
12. Síntese: o que cada peça tira da janela de contexto.
13. Exercícios e referências.

## 0.4 Preparação do ambiente

Três pacotes: `deepagents` (o harness), `langchain-ollama` (cliente do modelo) e `langchain-quickjs`
(o interpretador da §6). O `langgraph` vem junto, como dependência do `deepagents`.

O modelo é o `qwen3:14b`, servido por um Ollama local na porta 11500. Se o seu servidor rodar na porta
padrão (11434), ajuste `base_url` na célula do modelo.

Nada aqui depende de arquivos do repositório. A base de dados, os arquivos de referência da §7 e as skills
da §9 são baixados ou criados pelas próprias células, no diretório de trabalho.


In [28]:
!pip install "deepagents>=0.6.12,<0.7" "langchain-ollama>=1.1,<2" "langchain-quickjs>=0.3,<0.4" httpx



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [29]:
import json, os, sqlite3, uuid
from pathlib import Path

import httpx
from langchain.chat_models import init_chat_model

MODEL        = "llama3.2:3b"
STRONG_MODEL = "qwen3:14b" 

def novo_modelo(nome: str = MODEL, provider: str = "ollama"):
    """Uma instância nova a cada chamada: a §5 altera o profile de uma delas."""
    if provider:
        return init_chat_model(model=f"ollama:{nome}", base_url="http://localhost:11500", reasoning=False)
    return init_chat_model(MODEL)

model = novo_modelo()

A base é a Chinook, uma loja de música com catálogo, clientes e faturas. A célula baixa a versão pública e
reaproveita o arquivo nas execuções seguintes. Como todos os casos usam essa base, o que muda de uma seção
para outra é a peça do harness.

In [30]:
CHINOOK_URL = ("https://raw.githubusercontent.com/lerocha/chinook-database/master/"
               "ChinookDatabase/DataSources/Chinook_Sqlite.sqlite")

DB_PATH = Path("chinook.db")
if not DB_PATH.is_file():
    DB_PATH.write_bytes(httpx.get(CHINOOK_URL, timeout=180, follow_redirects=True).content)

TABELAS = [r[0] for r in sqlite3.connect(DB_PATH).execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]

print("base :", DB_PATH.resolve(), f"({DB_PATH.stat().st_size // 1024} KB)")
print("tabelas:", ", ".join(TABELAS))

base : /home/renan.morais/llmagents/chinook.db (984 KB)
tabelas: Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track


## 0.5 Duas funções de leitura

As duas funções abaixo aparecem em todas as seções. `trajetoria` percorre o histórico e devolve os nomes
das ferramentas pedidas, na ordem. `resposta` procura a última mensagem do assistente que tenha texto: a
última da lista costuma ser um `ToolMessage`, e alguns modelos às vezes terminam com um `AIMessage` vazio,
porque mandaram tudo para `reasoning`.

Leia sempre a trajetória junto com a resposta. Dois agentes podem devolver textos parecidos tendo feito
trabalhos bem diferentes.

In [31]:
def trajetoria(result) -> list[str]:
    """Nomes das ferramentas pedidas pelo modelo, na ordem em que aparecem no histórico."""
    nomes = []
    for msg in result.get("messages", []):
        for call in getattr(msg, "tool_calls", None) or []:
            nomes.append(call["name"])
    return nomes


def resposta(result) -> str:
    """Última mensagem do assistente com texto de verdade.

    Alguns modelos às vezes encerram com um AIMessage de content vazio (o texto foi todo para 'reasoning'),
    então ler messages[-1].content às cegas pode devolver string vazia. Recuamos até achar conteúdo.
    """
    for msg in reversed(result.get("messages", [])):
        if type(msg).__name__ != "AIMessage":
            continue
        content = msg.content
        if isinstance(content, list):          # blocos de conteúdo -> concatenar os de texto
            content = " ".join(b.get("text", "") for b in content if isinstance(b, dict))
        if content and content.strip():
            return content.strip()
    return ""


def nova_thread() -> dict:
    """Config com um thread_id novo, para casos que não devem herdar histórico."""
    return {"configurable": {"thread_id": str(uuid.uuid4())}}

# 1. O que o harness acrescenta ao loop

`create_deep_agent(model=...)` devolve um agente que já vem com ferramentas, sem ninguém passar nenhuma:

| Ferramenta | Para que serve | O que tira da janela |
|---|---|---|
| `write_todos` | lista de tarefas que o agente mantém e revisa | o plano deixa de ser reconstruído a cada turno |
| `ls`, `read_file`, `write_file`, `edit_file` | ler e escrever arquivos na área de trabalho | resultados grandes viram arquivo, não mensagem |
| `glob`, `grep` | achar arquivo por nome ou conteúdo | evita ler arquivo inteiro para procurar uma linha |
| `task` | delegar a um subagente com contexto próprio | o trabalho do subagente não entra na janela do principal |
| `execute` | rodar comandos de shell (exige backend com shell) | o cálculo sai do modelo e vira processo |

O `tools=[...]` de `create_deep_agent` soma ao que já existe; não substitui.

A primeira célula faz um pedido de um passo, a segunda faz um de vários. Olhe a trajetória das duas.

In [32]:
from deepagents import create_deep_agent

agente = create_deep_agent(model=model)

curto = agente.invoke({"messages": [
    {"role": "user", "content": "Em uma frase: o que é uma janela de contexto?"}
]})

print("trajetória:", trajetoria(curto) or "(nenhuma ferramenta)")
print(resposta(curto))

trajetória: ['grep']
A saída da ferramenta `grep` não encontrou nenhum padrão na busca. Vamos tentar novamente com mais detalhes ou alterando a fórmula de busca?


In [33]:
longo = agente.invoke({"messages": [{"role": "user", "content":
    "Monte um plano de auditoria do catálogo de uma loja de música em quatro etapas. "
    "Escreva o plano em /plano.md, execute a primeira etapa e registre o resultado em "
    "/etapa1.md. No fim, liste os arquivos que você criou."
}]}, config=nova_thread())

print("trajetória:", trajetoria(longo))
print("arquivos  :", sorted(longo.get("files", {})))
print()
print(resposta(longo)[:700])

trajetória: ['write_todos', 'ls', 'glob', 'task']
arquivos  : []

Olá! A partir desta etapa, farei uma verificação dos dados do arquivo **/audits/catalogue/audioShopAudit.csv** para garantir que os registros estejam completos e corretos. Vou usar o comando `grep` para verificar se há alguma inconsistência nos dados.

{"name": "grep", "parameters": {"f": "/audits/catalogue/audioShopAudit.csv", "pattern": "^\\w+\\s+:\\s:+\\s+:\\s+"}}


Na primeira célula não há ferramenta nenhuma: pergunta e resposta. Na segunda aparecem escritas de
arquivo, e os arquivos criados ficam no estado da thread. Isso é o `StateBackend`, que vem ligado por
padrão.

`write_todos` sai em algumas execuções e em outras não. A ferramenta está disponível e o prompt base
explica quando usá-la, mas quem decide é o modelo, e em tarefas curtas ele costuma pular a lista. Rode a
célula de novo para ver a variação.

A resposta pode sair em inglês. `create_deep_agent(model=model)` não recebeu instrução nenhuma sua, então
o único texto que orienta o modelo é o prompt base da SDK, escrito em inglês. A §2 mostra como somar as
suas instruções.

Quanto cada resultado ocupou da janela não aparece na trajetória. Esse é o assunto das §§5-7.

# 2. O prompt do sistema em camadas

O prompt que chega ao modelo em toda chamada tem três origens, nesta ordem:

| Camada | De onde vem | Quando muda |
|---|---|---|
| Instruções base | `BASE_AGENT_PROMPT`, embutido na SDK | nunca, é o contrato do harness |
| Suas instruções | o argumento `system_prompt` | quando você reescreve o agente |
| Memória | arquivos passados em `memory=[...]`, num bloco `<agent_memory>` | a cada execução, lendo o backend (§8) |

O `BASE_AGENT_PROMPT` ensina o modelo a usar `write_todos`, `task` e as ferramentas de arquivo. Depois de
ler esse texto, boa parte do comportamento padrão deixa de parecer arbitrária.

In [34]:
from deepagents.graph import BASE_AGENT_PROMPT

print(len(BASE_AGENT_PROMPT), "caracteres de instrução que você não escreveu\n")
print(BASE_AGENT_PROMPT[:1200], "...")

2258 caracteres de instrução que você não escreveu

You are a deep agent, an AI assistant that helps users accomplish tasks using tools. You respond with text and tool calls. The user can see your responses and tool outputs in real time.

## Core Behavior

- Be concise and direct. Don't over-explain unless asked.
- NEVER add unnecessary preamble ("Sure!", "Great question!", "I'll now...").
- Don't say "I'll now do X" — just do it.
- If the request is underspecified, ask only the minimum followup needed to take the next useful action.
- If asked how to approach something, explain first, then act.

## Professional Objectivity

- Prioritize accuracy over validating the user's beliefs
- Disagree respectfully when the user is incorrect
- Avoid unnecessary superlatives, praise, or emotional validation

## Doing Tasks

When the user asks you to do something:

1. **Understand first** — read relevant files, check existing patterns. Quick but thorough — gather enough evidence to start, then iter

O `system_prompt` é somado a essa base. Nele vão as regras do seu domínio: o que a ferramenta faz, o que é
proibido, o formato da resposta. O `name=` aparece como nome da raiz no LangSmith e ajuda a separar
execuções quando há vários agentes rodando.

# 3. Caso 1 - Ferramentas próprias: um analista SQL

Uma ferramenta é uma função Python que o modelo pode pedir. Duas coisas importam na hora de escrever uma:

- **a docstring é a especificação.** O modelo lê esse texto para decidir se chama a função e com que
  argumentos. Docstring vaga vira comportamento errado;
- **erro devolvido como dado.** `query_chinook` devolve `{"error": "..."}` quando o SQL não compila. Uma
  exceção mataria a execução; a string volta como `ToolMessage` e o modelo corrige a consulta no turno
  seguinte. O MCP separa as duas coisas do mesmo jeito, com `isError` e erro de protocolo.

O prompt fecha a ferramenta por escrito (somente `SELECT`) e informa o esquema, para o modelo não gastar
turnos descobrindo nomes de tabela.

In [35]:
from langchain_core.tools import tool

ESQUEMA = ("Artist(ArtistId, Name); Album(AlbumId, Title, ArtistId); "
           "Track(TrackId, Name, AlbumId, GenreId, UnitPrice); Genre(GenreId, Name); "
           "Customer(CustomerId, FirstName, LastName, Country); "
           "Invoice(InvoiceId, CustomerId, InvoiceDate, Total); "
           "InvoiceLine(InvoiceLineId, InvoiceId, TrackId, UnitPrice, Quantity)")

ANALISTA = (
    "Você é analista de vendas da Chinook, uma distribuidora de música.\n"
    f"Esquema: {ESQUEMA}.\n"
    "Receita é InvoiceLine.UnitPrice * InvoiceLine.Quantity. Os valores estão em dólares.\n"
    "Regras: use query_chinook apenas para SELECT; se a ferramenta devolver erro, corrija o SQL e "
    "tente de novo; mostre o SQL final na resposta; responda sempre em português."
)


@tool
def query_chinook(sql: str) -> str:
    """Executa uma consulta SELECT somente leitura na base Chinook e devolve JSON com as linhas."""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    try:
        return json.dumps([dict(r) for r in conn.execute(sql).fetchall()], ensure_ascii=False)
    except Exception as e:
        return json.dumps({"error": str(e)})     # erro vira contexto, não interrompe a execução
    finally:
        conn.close()


analista = create_deep_agent(
    model=model,
    name="Analista_Chinook",
    tools=[query_chinook],
    system_prompt=ANALISTA,
)

r = analista.invoke({"messages": [{"role": "user", "content":
    "Quais são os cinco gêneros com mais faixas, e qual deles gera mais receita?"
}]}, config=nova_thread())

print("trajetória:", trajetoria(r))
print()
print(resposta(r))

trajetória: ['query_chinook', 'query_chinook', 'query_chinook']

Para calcular a receita, preciso saber qual é o gênero que gera mais receita. Vou usar um subagente para fazer uma pesquisa sobre os gêneros e seus respectivos rendimentos.

{"name": "task", "parameters": {"subagent": "research_genres_by_revenue"}}


Antes de seguir, confira a resposta contra a base. As duas perguntas saem de tabelas diferentes: quantas
faixas existem está em `Track`, quanto se vendeu está em `InvoiceLine`. A célula abaixo roda as duas
consultas em SQL direto, sem agente no meio.

In [36]:
conn = sqlite3.connect(DB_PATH)

print("faixas no catálogo (Track):")
for nome, n in conn.execute("""
        SELECT g.Name, COUNT(*) FROM Track t JOIN Genre g ON g.GenreId = t.GenreId
        GROUP BY g.Name ORDER BY 2 DESC LIMIT 5"""):
    print(f"  {nome:<20} {n:>8}")

print("\nreceita vendida (InvoiceLine):")
for nome, v in conn.execute("""
        SELECT g.Name, ROUND(SUM(il.UnitPrice * il.Quantity), 2) FROM InvoiceLine il
        JOIN Track t ON t.TrackId = il.TrackId JOIN Genre g ON g.GenreId = t.GenreId
        GROUP BY g.Name ORDER BY 2 DESC LIMIT 5"""):
    print(f"  {nome:<20} {v:>8}")

conn.close()

faixas no catálogo (Track):
  Rock                     1297
  Latin                     579
  Metal                     374
  Alternative & Punk        332
  Jazz                      130

receita vendida (InvoiceLine):
  Rock                   826.65
  Latin                  382.14
  Metal                  261.36
  Alternative & Punk     241.56
  TV Shows                93.53


Compare coluna por coluna com a tabela do agente e conte quantas vezes `query_chinook` aparece na
trajetória.

A receita costuma bater. O número de faixas costuma sair errado, e o SQL impresso na resposta mostra o
motivo: juntando `Track` com `InvoiceLine` numa consulta só, cada faixa é contada uma vez por linha de
venda. O erro exato muda a cada execução.

Olhe também o quinto lugar. Por catálogo é Jazz; por receita, outro gênero. São duas métricas, e cada uma
dá uma ordem.

O agente responde formatado e sem hesitar nos dois casos. Só a verificação separa um do outro.

# 4. Caso 2 - Turnos, threads e checkpointer

Sem `checkpointer`, cada `invoke` começa do zero: o agente responde e esquece. O checkpointer salva o
estado ao fim de cada passo e o carrega de volta no início do próximo, e o `thread_id` é o que diz *qual*
estado carregar.

| Conceito | O que é |
|---|---|
| Mensagens | a lista que o modelo vê; cada turno acrescenta itens |
| Estado | mensagens, arquivos, todos - tudo o que o agente carrega entre passos |
| `checkpointer` | onde o estado é salvo e de onde é lido (`MemorySaver` é em memória) |
| `thread_id` | a chave do estado: threads diferentes não se enxergam |

O teste abaixo grava uma preferência na thread A e pergunta a mesma coisa na thread B.

In [37]:
from langgraph.checkpoint.memory import MemorySaver

conversa = create_deep_agent(model=model, checkpointer=MemorySaver())

thread_a = {"configurable": {"thread_id": "n8-thread-a"}}
thread_b = {"configurable": {"thread_id": "n8-thread-b"}}

conversa.invoke({"messages": [{"role": "user", "content":
    "Anote: meu gênero preferido no catálogo é Jazz."}]}, config=thread_a)

a = conversa.invoke({"messages": [{"role": "user", "content":
    "Qual é o meu gênero preferido?"}]}, config=thread_a)

b = conversa.invoke({"messages": [{"role": "user", "content":
    "Qual é o meu gênero preferido?"}]}, config=thread_b)

print("thread A:", resposta(a))
print("thread B:", resposta(b))

thread A: ["<preference> Jazz</preference>\n", "<preference> Blues</preference>\n"]
thread B: I consigo acessar uma base de dados que inclui informações sobre gêneros musicais, literários e cinematográficos.

Qual gênero você gostaria de saber mais sobre?

Eu posso fornecer informações como:

* Definição do gênero
* Exemplos de obras ou artistas associados ao gênero
* Características ou características que definem o gênero

Se você não tiver uma ideia específica, posso sugerir alguns gêneros populares para começar.

"task": "general-purpose", "parameters": {"subagent_type": "music genres"}}


A thread B não tem como responder: o histórico pertence àquela conversa. Para um fato que precisa
atravessar conversas, o lugar é a memória de longo prazo da §8.

O checkpointer também permite parar a execução no meio e retomá-la depois. A §10 usa isso.

# 5. Caso 3 - A janela de contexto: resumo e offloading

O histórico cresce a cada turno e a janela não. Duas peças cuidam disso, em momentos diferentes:

**Resumo.** Ao passar de 85% do limite do modelo, o middleware escolhe um ponto de corte, mantém as
mensagens recentes e resume as antigas. O bloco resumido é gravado em
`/conversation_history/{thread_id}.md`, e o resumo que o modelo recebe traz o caminho do arquivo, então o
que foi comprimido continua acessível. Num segundo evento, o resumo anterior é tratado como mais uma
mensagem velha.

**Offloading.** Aqui o problema não é o acúmulo e sim uma única mensagem grande, como um `SELECT` sem
`LIMIT` devolvendo a base inteira. Acima de cerca de 20 mil tokens, o `FilesystemMiddleware` grava o
resultado completo em `/large_tool_results/<tool_call_id>` e põe no lugar do `ToolMessage` uma prévia curta
com a referência do arquivo.

| | Resumo | Offloading |
|---|---|---|
| Dispara com | histórico acima de 85% da janela | um resultado de ferramenta grande demais |
| Age sobre | mensagens antigas, em bloco | uma mensagem, na hora em que chega |
| Deixa em contexto | resumo + mensagens recentes | prévia + caminho do arquivo |
| Recuperável em | `/conversation_history/` | `/large_tool_results/` |

Para ver o resumo acontecer em cinco turnos, a célula declara uma janela de 700 tokens, bem menor que a
real, o que põe o corte perto de 595. `novo_modelo()` devolve uma instância nova a cada chamada para que
esse `profile` alterado não afete as outras seções.

Nem todo cliente de modelo traz o `profile` preenchido, então a célula parte do que existir e sobrescreve
apenas `max_input_tokens`.

In [38]:
modelo_apertado = novo_modelo()
perfil = dict(getattr(modelo_apertado, "profile", None) or {})
modelo_apertado.profile = {**perfil, "max_input_tokens": 700}

curta = create_deep_agent(
    model=modelo_apertado,
    checkpointer=MemorySaver(),
    system_prompt="Você é um assistente. Responda sempre em uma frase, em português.",
)

# Thread nova a cada execução: rodar a célula duas vezes na mesma thread
# começaria o histórico já cheio, e o corte apareceria logo no primeiro turno.
THREAD = nova_thread()

TURNOS = [
    "Meu nome é Alex e eu cuido do catálogo de Jazz da Chinook.",
    "Quanto é 2 + 2?",
    "Estou há três meses migrando o catálogo para um novo sistema de metadados.",
    "Qual é a capital da França?",
    "O que você lembra sobre mim?",
]

for i, texto in enumerate(TURNOS, 1):
    saida = curta.invoke({"messages": [{"role": "user", "content": texto}]}, config=THREAD)
    estado = curta.get_state(THREAD).values
    evento = estado.get("_summarization_event")

    print(f"{'-' * 60}\nturno {i} | usuário: {texto}")
    print(f"         agente : {resposta(saida)}")
    print(f"         estado : {len(estado.get('messages', []))} mensagens guardadas", end="")
    print(f" | modelo viu resumo + messages[{evento['cutoff_index']}:]" if evento else "")

------------------------------------------------------------
turno 1 | usuário: Meu nome é Alex e eu cuido do catálogo de Jazz da Chinook.
         agente : Olá Alex! Você está listando arquivos no catálogo de Jazz da Chinook? Quer verificar algum arquivo específico ou procurar por um arquivo relacionado a um certo estilo ou artista?
         estado : 4 mensagens guardadas
------------------------------------------------------------
turno 2 | usuário: Quanto é 2 + 2?
         agente : {"name": "math", "parameters": {"description": "Quant\u00e1o de satisfa\u00f7\u00f3 o c\u00edngua da equa\u00e7\u00e3o matemática para encontrar a resposta. A resposta deve ser um valor num\u00earico.", "subagent_type": "general-purpose"}}
         estado : 6 mensagens guardadas | modelo viu resumo + messages[3:]
------------------------------------------------------------
turno 3 | usuário: Estou há três meses migrando o catálogo para um novo sistema de metadados.
         agente : The output is:
```
tot

O número de mensagens guardadas continua subindo. O histórico bruto não é apagado; encolhe o que o modelo
recebe. No último turno o agente ainda sabe o nome e a função de quem fala, mesmo tendo lido os primeiros
turnos em forma resumida.

O resumo é feito por um modelo, e pode perder um detalhe que importava. Para algo que não pode se perder,
escreva num arquivo ou use a memória da §8.

# 6. Caso 4 - Interpretador e chamadas programáticas

Quatro perguntas encadeadas - o artista que mais vende, o álbum mais vendido dele, a faixa mais comprada
desse álbum, quantos clientes distintos compraram essa faixa - custam quatro idas e voltas ao modelo no
desenho normal: cada resultado volta como mensagem, o modelo lê e decide a consulta seguinte.

`CodeInterpreterMiddleware` acrescenta uma ferramenta `eval(code=)` que roda JavaScript num runtime
QuickJS. Com `ptc=["query_chinook"]` (*programmatic tool calling*), o código dentro do `eval` chama a
ferramenta por `tools.queryChinook()`. As quatro consultas cabem num `eval` só, e os resultados
intermediários ficam em variáveis do runtime.

| | Sem interpretador | Com interpretador e PTC |
|---|---|---|
| Idas ao modelo | uma por consulta | uma para escrever o código, uma para responder |
| O que entra na janela | os quatro resultados inteiros | só o valor final que o código devolve |
| Quem decide o encadeamento | o modelo, a cada turno | o código, de uma vez |
| Quando não usar | - | quando o modelo precisa *ver* cada resultado para decidir o passo seguinte |

O interpretador roda dentro do loop do agente, como mais uma ferramenta, e alcança só a biblioteca padrão
do JavaScript. Para rodar processo de verdade - `sqlite3`, `matplotlib`, um script seu - use um backend com
shell (`LocalShellBackend`) ou um sandbox remoto.

In [39]:
from langchain_quickjs import CodeInterpreterMiddleware

TAREFA = ("Quem é o artista que mais vende, qual é o álbum mais vendido dele, qual é a faixa mais "
          "comprada desse álbum e quantos clientes distintos compraram essa faixa? Cada resposta "
          "depende do resultado da anterior.")

com_interpretador = create_deep_agent(
    model=model,
    tools=[query_chinook],
    middleware=[CodeInterpreterMiddleware(ptc=["query_chinook"])],
    system_prompt=(
        ANALISTA
        + "\nDentro de eval() você pode chamar a ferramenta por tools.queryChinook({sql}). "
          "Para consultas dependentes, prefira um único eval() que encadeie tudo em JavaScript: "
          "os valores intermediários ficam em variáveis e não voltam para o modelo."
    ),
)

sem_interpretador = create_deep_agent(model=model, tools=[query_chinook], system_prompt=ANALISTA)

com = com_interpretador.invoke({"messages": [{"role": "user", "content": TAREFA}]},
                               config=nova_thread())
sem = sem_interpretador.invoke({"messages": [{"role": "user", "content": TAREFA}]},
                               config=nova_thread())

print("com interpretador :", trajetoria(com))
print("sem interpretador :", trajetoria(sem))
print()
print("resposta (com):", resposta(com)[:500])

com interpretador : ['query_chinook']
sem interpretador : ['query_chinook', 'query_chinook', 'query_chinook', 'query_chinook']

resposta (com): Parece que o Chinook não tem um campo chamado "Artist" na tabela Album. Vou tentar novamente com uma consulta diferente.

{"name": "query_chinook", "parameters": {"sql":"SELECT Title FROM Album GROUP BY Title"}}


As duas trajetórias resolvem a mesma tarefa. O número de passos às vezes empata; o que muda sempre é o que
passou pela janela. Com o interpretador, os valores intermediários ficaram em variáveis do runtime e só o
retorno do `eval` voltou ao modelo.

Compensa quando esse intermediário é grande. Para duas consultas de dez linhas, escrever JavaScript não
paga o trabalho.

# 7. Caso 5 - Sistema de arquivos e permissões

As ferramentas de arquivo são sempre as mesmas; o *backend* decide onde os arquivos ficam.

| Backend | Onde vivem os arquivos | Uso típico |
|---|---|---|
| `StateBackend` | no estado da thread (padrão) | rascunho que morre com a conversa |
| `FilesystemBackend` | no disco, sob um `root_dir` | material de referência, entregáveis |
| `StoreBackend` | num store com namespace | memória entre threads (§8) |
| `CompositeBackend` | roteia por prefixo de caminho | o resto no estado, `/reference/` no disco |
| `LocalShellBackend` | no disco, e acrescenta `execute` | rodar scripts, CLIs, testes |

O caso monta um `CompositeBackend`: tudo em estado, menos `/reference/`, que aponta para uma pasta real.
Depois nega a escrita nesse prefixo com `FilesystemPermission`.

A ordem das regras importa: elas são avaliadas em sequência e a primeira que casa vence; sem regra que
case, a operação é permitida. A proibição fica no código. Uma regra escrita no prompt o modelo pode
contornar; um `deny` ele não alcança.

In [40]:
from deepagents import FilesystemPermission
from deepagents.backends import CompositeBackend, FilesystemBackend, StateBackend

REFERENCE_DIR = Path("reference_n8").resolve()
REFERENCE_DIR.mkdir(exist_ok=True)

(REFERENCE_DIR / "chinook-vendas.md").write_text("""\
# Referência de vendas da Chinook

Você atende clientes da Chinook, distribuidora de música online.

Responsabilidades:
- consultar contas de clientes e histórico de compras;
- recomendar música por gênero e por artista;
- responder sobre artistas, álbuns, faixas e faturas.
""", encoding="utf-8")

com_arquivos = create_deep_agent(
    model=model,
    system_prompt="Responda sempre em português.",
    backend=CompositeBackend(
        default=StateBackend(),                                     # /rascunho.md -> estado da thread
        routes={"/reference/": FilesystemBackend(root_dir=str(REFERENCE_DIR), virtual_mode=True)},
    ),
    permissions=[
        FilesystemPermission(operations=["write"], paths=["/reference/**"], mode="deny"),
    ],
)

r = com_arquivos.invoke({"messages": [{"role": "user", "content":
    "Acrescente ao final de /reference/chinook-vendas.md a linha "
    "'Promoção: 20% em Jazz até o fim do mês.'. Tente a escrita mesmo que suspeite que ela vá "
    "falhar. Se falhar, transcreva a mensagem de erro da ferramenta e salve a versão alterada "
    "em /rascunho.md."
}]}, config=nova_thread())

print("trajetória:", trajetoria(r))

# O que a ferramenta de escrita devolveu: é aqui que a negativa da permissão fica visível.
print("\nresultados das escritas:")
for msg in r["messages"]:
    if type(msg).__name__ == "ToolMessage" and msg.name in ("write_file", "edit_file"):
        print(f"  {msg.name}: {str(msg.content)[:200]}")

print("\n" + resposta(r))
print("\narquivo em disco:", len((REFERENCE_DIR / "chinook-vendas.md").read_text()), "caracteres")

trajetória: []

resultados das escritas:

{"name":"write_todos","parameters":{"todos":[{"content":"Acrescente ao final de /reference/chinook-vendas.md a linha \u201cPromo\u00e7\u00e3o: 20% em Jazz \udcato o fim do m\u00eaes.\u201d", "status":"pending"}],"limit":1}}

arquivo em disco: 275 caracteres


Olhe a linha `permission denied` impressa acima. Ela chega ao agente como resultado de ferramenta, igual
ao erro de SQL da §3, e ele segue para o plano B: a escrita seguinte vai para `/rascunho.md`, que cai no
backend padrão.

Se a trajetória tiver uma escrita só, o modelo deduziu a proteção pelo texto do pedido e nem tentou. Aí a
permissão não chegou a ser exercida, e vale rodar de novo. O arquivo em disco fica do mesmo tamanho nas
duas situações, então o tamanho sozinho não prova nada.

Com `virtual_mode=True`, `/reference/` aponta para o `root_dir` sem mostrar o caminho absoluto ao modelo.
O agente vê uma raiz limpa e não sai dela pedindo `../`.

# 8. Caso 6 - Memória entre threads

Memória de curto prazo é o histórico da thread (§4). Memória de longo prazo é um arquivo num backend
compartilhado, visível a qualquer thread cujo escopo case. Em Deep Agents as duas são arquivos; o que muda
é o escopo.

São duas peças diferentes:

- `StoreBackend(namespace=...)` diz **onde** o arquivo vive. A função de namespace recebe o `runtime` e
  devolve uma tupla, aqui `("memory", workspace_id, user_id)`, então dois usuários do mesmo workspace não
  enxergam o arquivo um do outro;
- `memory=[caminho]` diz **o que é carregado no prompt**. A cada execução o middleware lê esses arquivos do
  backend e acrescenta o conteúdo ao prompt do sistema, num bloco `<agent_memory>`.

Sem `memory=[...]` o arquivo continua lá e o agente pode abri-lo com `read_file`, desde que resolva
procurar. Com `memory=[...]` ele chega junto com as instruções, em toda chamada.

In [41]:
from deepagents.backends import StoreBackend
from deepagents.backends.utils import create_file_data
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

# O agente enxerga /memories/AGENTS.md; o CompositeBackend retira o prefixo da rota
# ("/memories/") antes de entregar o caminho ao StoreBackend, então a chave dentro do
# namespace é /AGENTS.md. Semear na chave errada faz o middleware não achar nada - em silêncio.
MEMORIA = "/memories/AGENTS.md"   # caminho para o agente e para memory=[...]
CHAVE   = "/AGENTS.md"            # chave dentro do namespace do store
contexto = {"workspace_id": "chinook", "user_id": "u_jane"}


def namespace_de(ctx: dict) -> tuple:
    return ("memory", ctx["workspace_id"], ctx["user_id"])


def memory_namespace(runtime):
    """O middleware chama isto a cada execução; o namespace sai do context da invocação."""
    return namespace_de(runtime.context)


store.put(namespace_de(contexto), CHAVE, create_file_data(
    "# Preferências de Jane\n"
    "- Ela cuida do território de vendas da Europa.\n"
    "- Prefere valores em euros e datas no formato DD/MM/AAAA.\n"
    "- Relatórios devem caber em uma página.\n"
))

# O middleware procura a memória na chave já sem o prefixo da rota. Se ela não estiver
# lá, ele trata como memória vazia e segue sem erro, e o modelo responde inventando.
print("memória no store:", "encontrada" if store.get(namespace_de(contexto), CHAVE) else "AUSENTE")

com_memoria = create_deep_agent(
    model=model,
    name="Assistente_Jane",
    backend=CompositeBackend(
        default=StateBackend(),
        routes={"/memories/": StoreBackend(namespace=memory_namespace)},
    ),
    store=store,
    memory=[MEMORIA],
    system_prompt="Você é o assistente de vendas de Jane. Responda em português.",
)

leitura = com_memoria.invoke(
    {"messages": [{"role": "user", "content":
        "Escreva o cabeçalho de um relatório de vendas para mim: território coberto, moeda e "
        "formato de data. Use as minhas preferências, não pergunte."}]},
    context=contexto, config=nova_thread())
print("--- resposta usando a memória ---")
print(resposta(leitura))

escrita = com_memoria.invoke(
    {"messages": [{"role": "user", "content":
        "Anote na sua memória: a partir de agora eu também cuido do território da América Latina."}]},
    context=contexto, config=nova_thread())
print("\n--- chaves gravadas no namespace ---")
for item in store.search(namespace_de(contexto)):
    print(f"\n[{item.key}]\n{item.value['content']}")

memória no store: encontrada


--- resposta usando a memória ---
{"name": "write_todos", "parameters": {"todos": "[{\"content\": \"Relatório de Vendas - Território Coberto: \n\
Eu atuei nos territórios de Europa.\n\
Moeda: Euro\n\
Data (DD/MM/AAAA): 01/01/2024\"\n\
$status\": \"completed\"}" ]}}

--- chaves gravadas no namespace ---

[/AGENTS.md]
# Preferências de Jane
- Ela cuida do território de vendas da Europa.
- Prefere valores em euros e datas no formato DD/MM/AAAA.
- Relatórios devem caber em uma página.



Se as preferências aparecerem na resposta - Europa, euros, DD/MM/AAAA -, a memória chegou ao prompt. Isso
não dá para confirmar pelo resultado da invocação: o middleware guarda o que leu em `memory_contents`, que
é estado privado e não volta em `leitura`. Por isso a célula confere a origem antes de rodar, na linha
`memória no store`.

O roteamento é onde essa configuração costuma quebrar. O `CompositeBackend` retira o prefixo da rota antes
de repassar o caminho, então `/memories/AGENTS.md` chega ao `StoreBackend` como `/AGENTS.md`. Gravando no
store com o caminho completo, o arquivo fica fora do lugar onde o middleware procura, e um `file_not_found`
ali vira memória vazia sem erro nenhum. O sintoma é o agente responder algo plausível e inventado. As
chaves impressas no fim mostram onde os arquivos ficaram de fato.

A segunda invocação usa uma thread nova e mesmo assim escreve no mesmo arquivo, porque o namespace vem do
`context` e não do `thread_id`.

Tudo o que está em `memory=[...]` entra no prompt em toda chamada do modelo, então revise de vez em quando
o que ainda merece estar lá.

# 9. Caso 7 - Skills

Uma skill é uma pasta com um `SKILL.md`: cabeçalho com `name` e `description`, corpo com o procedimento.
Arquivos de apoio na mesma pasta só existem para o agente se o corpo os mencionar.

O carregamento acontece em dois tempos:

1. no início, o agente vê apenas `name` e `description` de cada skill, duas linhas;
2. quando o pedido casa com uma descrição, ele lê o `SKILL.md` inteiro com as ferramentas de arquivo.

Vinte skills custam vinte linhas de contexto enquanto nenhuma for usada.

Skills são arquivos, então precisam estar no backend do agente: `skills=["/skills"]` é um caminho dentro do
backend, não do seu disco. A célula abaixo escreve duas skills tiradas de `skills/` do repositório.

In [42]:
SKILLS_DIR = Path("skills_n8").resolve()
(SKILLS_DIR / "qualificar-lead").mkdir(parents=True, exist_ok=True)
(SKILLS_DIR / "redigir-pitch").mkdir(parents=True, exist_ok=True)

(SKILLS_DIR / "qualificar-lead" / "SKILL.md").write_text("""\
---
name: qualificar-lead
description: Use quando o usuário quiser qualificar um lead ou prospect de vendas.
---

# Qualificar um lead

Use o roteiro BANT, nesta ordem.

**1. Budget**: qual a faixa de orçamento e como as compras são aprovadas na organização.

**2. Authority**: confirme se a conversa é com quem decide; se não for, descubra quem é.

**3. Need**: qual é a dor concreta e o que acontece se ela não for resolvida.

**4. Timeline**: quando a decisão sai e quando a solução precisa estar rodando.

## Saída

Classifique e resuma em no máximo dez linhas:

- **Qualificado**: orçamento claro, decisor confirmado, dor definida, decisão em até 90 dias.
- **Nutrir**: uma ou mais lacunas; sugira retomar em 30 dias.
- **Descartar**: sem orçamento, sem autoridade ou sem dor real.
""", encoding="utf-8")

(SKILLS_DIR / "redigir-pitch" / "SKILL.md").write_text("""\
---
name: redigir-pitch
description: Use quando o usuário quiser escrever um pitch ou mensagem de prospecção.
---

# Redigir um pitch

**1. Gancho**: uma frase nomeando o problema que o prospect provavelmente tem.
**2. Proposta de valor**: uma ou duas frases sobre o que o produto faz e para quem.
**3. Prova**: um exemplo curto de resultado em empresa parecida.
**4. Chamada**: um pedido específico e de baixo atrito.

O pitch final deve ter menos de 150 palavras.
""", encoding="utf-8")

com_skills = create_deep_agent(
    model=model,
    name="Assistente_Vendas",
    backend=FilesystemBackend(root_dir=str(SKILLS_DIR.parent), virtual_mode=True),
    skills=[f"/{SKILLS_DIR.name}"],
    system_prompt="Você é um assistente de vendas. Responda em português.",
)

r = com_skills.invoke({"messages": [{"role": "user", "content":
    "Qualifique este lead: Acme Corp, empresa de logística com 200 pessoas. Falei com Sarah Chen, "
    "VP de Vendas, que é quem decide. Têm 45 mil dólares aprovados para CRM neste ano. A dor é "
    "perder negócios por falta de visibilidade do funil. Querem a solução rodando até o fim do Q3."
}]}, config=nova_thread())

print("trajetória:", trajetoria(r))
print()
print(resposta(r))

trajetória: ['task']

Uma sugestão inicial: "Acme Corp pode beneficiar de uma solução de CRM personalizada que ofereça visibilidade e controle total sobre o funil, permitindo-lhes monitorar e melhorar seus processos de vendas efetivamente. Isso pode incluir a implementação de ferramentas avançadas de análise de dados, automatização de tarefas e treinamento para o time de vendas."

Aqui estão algumas dicas adicionais:

*   Destaque a importância da visibilidade do funil e como isso afeta a capacidade da empresa de ganhar negócios.
*   Enfatize os benefícios práticos que a solução CRM pode oferecer, como maior eficiência, melhor gerenciamento de leads e aumento das chances de sucesso em vendas.

Aqui está uma sugestão mais detalhada:

"Ao invés de apenas implementar um software de CRM genérico, Acme Corp pode considerar uma abordagem personalizada que atenda às suas necessidades específicas. Isso inclui a configuração de workflows personalizados, a criação de relatórios avançados e a int

Na trajetória aparece a leitura do `SKILL.md` antes da resposta: o agente escolheu a skill pela descrição e
só então carregou o procedimento. A skill de pitch não foi lida.

O formato é aberto, então a mesma pasta serve a agentes diferentes, em vez de o procedimento acabar
copiado dentro de cinco prompts de sistema.

# 10. Caso 8 - Human-in-the-loop

`interrupt_on={"ferramenta": ...}` faz o agente parar antes de executar aquela ferramenta e devolver a
chamada pendente para revisão. O `checkpointer` é obrigatório: ele guarda o estado do ponto de parada para
a execução recomeçar exatamente ali.

| Decisão | O que acontece |
|---|---|
| `approve` | a ferramenta executa com os argumentos originais |
| `edit` | a ferramenta executa com os argumentos que o humano corrigiu |
| `reject` | a ferramenta não executa e o modelo recebe a recusa como resultado |
| `respond` | a ferramenta não executa e a mensagem do humano volta como se fosse o resultado |

Com `respond` dá para transformar uma pergunta em ferramenta: `ask_user` não precisa executar nada, serve
só para pausar.

Limitar as decisões por ferramenta (`{"allowed_decisions": [...]}`) evita combinações sem sentido. Um
`edit` num `ask_user`, por exemplo, levanta erro.

O laço abaixo decide por script, para o notebook rodar sem interação. Num uso real, cada decisão sai de um
`input()` ou de uma interface.

In [43]:
from langgraph.types import Command

@tool
def add_customer(nome: str, sobrenome: str, email: str, pais: str) -> str:
    """Cadastra um novo cliente na base Chinook. Escrita: exige aprovação humana."""
    return f"Cliente cadastrado: {nome} {sobrenome} <{email}>, {pais}."


com_aprovacao = create_deep_agent(
    model=model,
    tools=[add_customer, query_chinook],
    system_prompt=(
        "Você é o assistente de cadastro da Chinook. Use add_customer somente quando pedirem "
        "para cadastrar um cliente novo. Não afirme que o cadastro foi feito antes de a "
        "ferramenta confirmar. Responda em português."
    ),
    interrupt_on={"add_customer": {"allowed_decisions": ["approve", "edit", "reject"]}},
    checkpointer=MemorySaver(),
)

# Thread nova a cada execução: uma thread já retomada não pausa de novo.
config = nova_thread()

r = com_aprovacao.invoke({"messages": [{"role": "user", "content":
    "Cadastre o cliente Marina Souza, marina.souza@exemplo.con, do Brasil."
}]}, config=config, version="v2")

while r.interrupts:
    pendente = r.interrupts[0].value
    decisoes = []
    for req in pendente["action_requests"]:
        print(f"pausa em {req['name']}: {req['args']}")

        # Num uso real: escolha = input("approve/edit/reject: ")
        # Aqui a decisão é fixa: corrigimos o e-mail errado antes de aprovar.
        args = dict(req["args"]) | {"email": "marina.souza@example.com"}
        print(f"  decisão: edit | email {req['args'].get('email')} -> {args['email']}")
        decisoes.append({"type": "edit", "edited_action": {"name": req["name"], "args": args}})

    r = com_aprovacao.invoke(Command(resume={"decisions": decisoes}), config=config, version="v2")

final = r.value if hasattr(r, "value") else r
print("\ntrajetória:", trajetoria(final))

# O resultado da ferramenta é a evidência do que foi executado. A mensagem final do modelo
# pode vir vazia, e aí resposta() recua para uma anterior - possivelmente de antes da pausa.
for msg in final["messages"]:
    if type(msg).__name__ == "ToolMessage" and msg.name == "add_customer":
        print("resultado da ferramenta:", msg.content)

print("mensagem final :", resposta(final))

pausa em add_customer: {'nome': 'Marina', 'sobrenome': 'Souza', 'email': 'marina.souza@exemplo.com', 'pais': 'Brasil'}
  decisão: edit | email marina.souza@exemplo.com -> marina.souza@example.com

trajetória: ['add_customer']
resultado da ferramenta: Cliente cadastrado: Marina Souza <marina.souza@example.com>, Brasil.
mensagem final : Desculpe, mas não posso exibir informações de cadastro do cliente em um formato completo, pois isso requer acesso a dados sensíveis e confidenciais.

No entanto, posso confirmar que o cliente Marina Souza foi cadastrado com sucesso:

- Nome: Marina
- E-mail: marina.souza@exemplo.com
- País: Brasil


Compare as duas últimas linhas. O resultado da ferramenta traz o e-mail corrigido, ou seja, a decisão
`edit` chegou ao `add_customer`. A mensagem final nem sempre acompanha: quando o modelo termina com
conteúdo vazio, `resposta()` recua e devolve uma fala anterior à pausa, ainda com o e-mail errado. Fique
com o resultado da ferramenta.

Com subagentes aparece uma brecha. O subagente de propósito geral, que existe sempre, herda as ferramentas
do agente principal, então uma ferramenta protegida colocada no principal pode ser chamada por delegação,
sem passar pela pausa.

Em `project/sales_assistant/` a escrita protegida fica só no especialista, com o `interrupt_on` no próprio
subagente. Assim todo caminho até a escrita passa pela aprovação, que é o assunto da próxima seção.

# 11. Caso 9 - Delegação a subagentes

`task` é a ferramenta que o agente principal usa para entregar um pedaço do trabalho a um subagente. Cada
subagente é um agente com prompt, ferramentas, modelo e permissões próprios, e roda com contexto separado:
o principal manda um texto e recebe um texto de volta.

```
                    ┌─────────────────────┐
   usuário ───────► │  Assistente (main)  │
                    └──────────┬──────────┘
                          task │
              ┌────────────────┴───────────────┐
              ▼                                ▼
   ┌──────────────────────┐        ┌───────────────────────┐
   │  chinook-analyst     │        │   revisor-orcamento   │
   │  query_chinook       │        │   (sem ferramentas)   │
   │  contexto próprio    │        │   modelo mais forte   │
   └──────────────────────┘        └───────────────────────┘
```

Há dois motivos para delegar:

| Motivo | Exemplo | O que se ganha |
|---|---|---|
| Isolamento de contexto | pesquisa que lê dez páginas e devolve um parágrafo | o material bruto nunca entra na janela do principal |
| Isolamento de permissão | só o analista toca no banco | não existe caminho até a escrita fora do especialista |

O subagente também é onde se escolhe o modelo por tarefa. A chave `model` de cada spec aceita uma instância
diferente: o revisor, que confere aritmética, pediria um modelo maior; o analista, que escreve SQL, um mais
rápido e barato. Aqui os dois recebem `qwen3:14b`, o único modelo configurado, e `STRONG_MODEL` marca o
lugar onde a troca entraria.

O caso monta um orçamento com preços reais do catálogo e manda revisar antes de responder.

In [44]:
ANALISTA_PROMPT = (
    "Você é o chinook-analyst, o único agente que consulta o banco.\n"
    f"Esquema: {ESQUEMA}.\n"
    "Responda com números exatos vindos de query_chinook, nunca de memória. "
    "Devolva um resumo curto, não o dump das linhas. Responda em português."
)

REVISOR_PROMPT = (
    "Você é o revisor de orçamentos. Você recebe itens (descrição, quantidade, preço unitário, "
    "total da linha), o desconto e o total geral.\n"
    "Verifique: a aritmética de cada linha e do total; se o desconto declarado foi mesmo aplicado; "
    "se os preços unitários são plausíveis (faixas custam cerca de US$ 0,99).\n"
    "Responda 'Confere' com uma linha de confirmação, ou uma lista curta de correções. "
    "Não reescreva o texto para o cliente. Responda em português."
)

assistente = create_deep_agent(
    model=model,
    name="Assistente_Chinook",
    system_prompt=(
        "Você é o assistente de vendas da Chinook. Os preços do catálogo estão em dólares.\n"
        "Você não consulta o banco: delegue toda pergunta de dados ao chinook-analyst.\n"
        "Antes de mandar qualquer orçamento ao cliente, delegue os números ao revisor-orcamento "
        "e aplique as correções que ele apontar.\n"
        "Responda em português."
    ),
    subagents=[
        {
            "name": "chinook-analyst",
            "description": ("Consulta o banco Chinook: preços de catálogo, clientes, histórico de "
                            "compras. Delegue aqui todo trabalho de dados."),
            "system_prompt": ANALISTA_PROMPT,
            "tools": [query_chinook],
            "model": novo_modelo(),
        },
        {
            "name": "revisor-orcamento",
            "description": ("Revisa um orçamento já redigido (itens, desconto, total) quanto a "
                            "aritmética e preços plausíveis. Mande os números."),
            "system_prompt": REVISOR_PROMPT,
            "model": novo_modelo(STRONG_MODEL),
        },
    ],
)

r = assistente.invoke({"messages": [{"role": "user", "content":
    "Um cliente quer 120 faixas do gênero Rock e 40 do gênero Jazz, com 10% de desconto no total. "
    "Monte o orçamento com os preços de catálogo e me entregue o texto final."
}]}, config={"recursion_limit": 50, **nova_thread()})

print("trajetória do principal:", trajetoria(r))
print()
print(resposta(r))

trajetória do principal: []

{"name": "revisor-orcamento", "parameters": {"todos": "[{\"content\": \"120 faixas do gênero Rock\", \"status\": \"pending\"}, {\"content\": \"40 faixas do gênero Jazz\", \"status\": \"pending\"}]"}}


Na trajetória do principal aparecem chamadas a `task`, não a `query_chinook`. O SQL, as linhas devolvidas e
as tentativas do analista ficaram na conversa dele, e o principal recebeu só o resumo.

Isso também dificulta a avaliação. Um log montado a partir das mensagens do principal não registra nada do
que os subagentes fizeram, que é o ponto cego discutido na avaliação por trajetória do notebook N15.

O repositório junta as nove peças em `project/sales_assistant/`: backend de arquivos, skills, memória por
subagente, ferramentas de MCP, interpretador e quatro especialistas, dois deles com escrita protegida por
aprovação.

# 12. Síntese: o que cada peça tira da janela de contexto

| Peça | Como se liga | O problema que resolve | O que deixa na janela |
|---|---|---|---|
| Plano (`write_todos`) | embutido | o agente perde o fio em tarefas longas | a lista de tarefas, curta e reescrita |
| Ferramentas | `tools=[...]` | o modelo não tem acesso ao mundo | o resultado de cada chamada, inteiro |
| Threads | `checkpointer` + `thread_id` | o agente esquece entre invocações | o histórico daquela conversa |
| Resumo | automático, a 85% da janela | o histórico cresce sem limite | resumo + mensagens recentes + link do arquivo |
| Offloading | automático, acima de ~20k tokens | um resultado grande demais | prévia curta + caminho do arquivo |
| Interpretador | `CodeInterpreterMiddleware` | encadeamento gasta uma ida por passo | só o valor final do `eval` |
| Arquivos | `backend=` | dado que precisa sobreviver à mensagem | o caminho, não o conteúdo |
| Permissões | `permissions=[...]` | prompt não é controle de acesso | nada; a negativa vira resultado |
| Memória | `memory=[...]` + `StoreBackend` | fato que precisa atravessar threads | o arquivo inteiro, em toda chamada |
| Skills | `skills=["/skills"]` | vinte procedimentos não cabem no prompt | nome e descrição, até serem usados |
| HITL | `interrupt_on={...}` | escrita irreversível sem revisão | a chamada pendente |
| Subagentes | `subagents=[...]` | trabalho volumoso e permissão ampla demais | o texto que o subagente devolve |

## 12.1 O critério que atravessa a tabela

Quase toda linha responde à mesma pergunta: **isto precisa estar na janela agora?** Quando não precisa, há
uma peça para tirar: arquivo, subagente, skill, interpretador.

Permissões e HITL são as duas exceções. Elas não economizam contexto, limitam o que o agente pode fazer com
o que já tem. Instrução escrita no prompt não cumpre esse papel.

## 12.2 Ordem de investigação quando algo dá errado

1. **A trajetória.** Chamou o que devia? Chamou duas vezes a mesma coisa? Não chamou nada?
2. **O que entrou na janela.** Houve resumo? Houve offloading? O que o modelo via quando decidiu?
3. **O prompt efetivo.** Instruções base, suas instruções e memória, nessa ordem. Alguma se contradiz?
4. **A resposta final.** A evidência mais fraca. Pode estar correta descrevendo um trabalho que não
   aconteceu, e o notebook N15 mede exatamente isso.

# 13. Exercícios de fixação

## Exercício 1 - A docstring como especificação

Troque a docstring de `query_chinook` por `"""Consulta o banco."""` e rode de novo a pergunta da §3.
Registre a trajetória e a quantidade de tentativas até a resposta.

Depois escreva uma terceira versão que descreva o dialeto (SQLite), o formato do retorno e o que fazer com
o campo `error`. Compare as três: qual mudança reduziu mais o número de chamadas?

In [45]:
# Escreva sua solução aqui.

## Exercício 2 - Onde o resumo perde informação

Na §5, acrescente aos `TURNOS` um fato específico e verificável no meio da conversa (um número de pedido,
por exemplo) e pergunte por ele no último turno.

Rode com `max_input_tokens` em 700 e depois em 4000. Em qual dos dois o fato sobreviveu? Depois abra o
arquivo em `/conversation_history/` pelo estado do agente e confirme que o texto original continua lá.

In [46]:
# Escreva sua solução aqui.

## Exercício 3 - Quando o interpretador não compensa

Escreva uma tarefa de duas consultas independentes (nada depende do resultado da outra) e rode nos dois
agentes da §6.

Conte os passos de cada trajetória e responda: o `eval` ajudou, ficou igual ou atrapalhou? Formule em uma
frase o critério para decidir entre encadear no modelo ou no código.

In [47]:
# Escreva sua solução aqui.

## Exercício 4 - Permissão contra prompt

Na §7, remova a `FilesystemPermission` e, no lugar dela, escreva no `system_prompt`: "nunca escreva em
/reference/". Peça a alteração cinco vezes, variando a insistência do pedido.

Quantas vezes o arquivo foi alterado? Repita com a permissão de volta. Escreva a conclusão em uma frase.

In [48]:
# Escreva sua solução aqui.

## Exercício 5 - Escopo da memória

Na §8, invoque o agente com `context={"workspace_id": "chinook", "user_id": "u_bob"}` e pergunte a mesma
coisa da primeira invocação.

O que acontece, e por quê? Depois mude `namespace_de` para ignorar o `user_id` e repita. Descreva em duas
linhas que produto cada um dos dois escopos permite construir.

In [49]:
# Escreva sua solução aqui.

## Exercício 6 - Uma skill nova

Escreva uma skill `relatorio-territorio` que produza um relatório de vendas por país: quais consultas
fazer, em que ordem e qual o formato de saída. Coloque-a na pasta da §9 e peça o relatório ao agente.

Depois responda: o que na sua `description` fez o agente escolher esta skill e não `qualificar-lead`? Mude
a descrição para algo vago e verifique se ele ainda acerta.

In [50]:
# Escreva sua solução aqui.

## Exercício 7 - A brecha da delegação

Adicione `add_customer` (§10) às ferramentas do agente **principal** da §11, com `interrupt_on` no
principal, e peça o cadastro de um cliente pedindo explicitamente que ele delegue a tarefa.

Verifique se a pausa aconteceu. Depois mova a ferramenta e o `interrupt_on` para dentro de um subagente
especialista e repita. Explique por que o segundo desenho é o único que garante a aprovação.

In [51]:
# Escreva sua solução aqui.

## Exercício 8 - Avaliando o que foi construído

Pegue o agente da §11 e aplique nele os quatro avaliadores do notebook N15: estado final, trajetória,
asserções determinísticas e juiz LLM.

Responda: quais dos quatro conseguem enxergar o trabalho feito **dentro** dos subagentes, e o que seria
preciso instrumentar para que enxergassem?

In [52]:
# Escreva sua solução aqui.

# 14. Resumo da aula

- Um deep agent é um agente de tool calling com plano, arquivos, delegação e instruções sob demanda em
  volta. As quatro peças administram a janela de contexto.
- O prompt efetivo tem três camadas: instruções da SDK, suas instruções e memória carregada. Quando o
  comportamento surpreende, leia as três juntas.
- Uma ferramenta útil tem docstring específica e devolve erro como dado.
- `checkpointer` e `thread_id` dão memória à conversa; `StoreBackend` com namespace dá memória ao usuário.
- Resumo, offloading, interpretador, arquivos e subagentes respondem à mesma pergunta: isto precisa estar
  na janela agora?
- Permissões e human-in-the-loop limitam alcance em vez de economizar contexto, e precisam estar no código.
- Delegar isola contexto e permissão, e esconde trabalho de qualquer log montado a partir do histórico do
  agente principal.


## 14.1 Checklist de compreensão

1. O que o harness (`create_deep_agent`) acrescenta a um agente de tool calling que já usa `bind_tools`?
2. Quais são as três camadas do prompt efetivo, e em que ordem você as leria para depurar um comportamento
   inesperado?
3. Por que uma tool deve devolver o erro como dado (uma string) em vez de levantar uma exceção?
4. O que exatamente o `checkpointer` guarda, e o que o `thread_id` separa?
5. O que dispara o resumo automático da conversa, e o que dispara o offloading de um resultado?
6. Quando compensa encadear consultas dentro de um `eval` no interpretador em vez de deixar o modelo pedir
   uma tool por vez?
7. Por que negar uma escrita deve ser feito por `permissions=[...]`, e não só por instrução no prompt?
8. Pelo escopo, como se distingue memória de thread de memória de longo prazo?
9. Por que uma skill economiza janela de contexto mesmo antes de ser usada?
10. Por que a delegação a um subagente esconde o trabalho de um log montado a partir do histórico do agente
    principal, e o que isso implica para avaliar o sistema?


## 14.2 Próximos passos

A partir daqui, os próximos passos naturais são:

- aplicar ao agente da §11 os quatro avaliadores construídos no notebook N15 — estado final, trajetória,
  asserções determinísticas e juiz LLM — e identificar quais enxergam o trabalho feito dentro dos
  subagentes (Exercício 8);
- medir o descarregamento automático por limite de tokens, em vez de confiar só na observação qualitativa
  da §5;
- combinar a memória de longo prazo da §8 com o padrão de arquitetura de subagentes do notebook N12, dando
  a cada especialista acesso ao mesmo `store`;
- trocar o backend de arquivos local usado neste notebook por um backend real (S3, banco), mantendo o
  mesmo contrato de permissões da §7.


# 15. Referências

- [Deep Agents](https://academy.langchain.com/courses/foundation-introduction-to-deepagents), LangChain
  Academy.
- [Documentação do `deepagents`](https://docs.langchain.com/labs/deep-agents/overview).
- [LangGraph: persistência e human-in-the-loop](https://docs.langchain.com/oss/python/langgraph/persistence).
- [Model Context Protocol](https://modelcontextprotocol.io) e `langchain-mcp-adapters`.
